In [1]:
import pandas as pd

# =========================
# 1. Load datasets
# =========================
main_df = pd.read_csv('gaming_mental_health_10M_40features.csv')

students_df = pd.read_csv('Students Social Media Addiction.csv')
coffee_df = pd.read_csv('synthetic_coffee_health_10000 .csv')
health_df = pd.read_csv('health_lifestyle_classification.csv')

# =========================
# 2. Standardize column names
# =========================
def standardize_columns(df):
    df.columns = df.columns.str.strip().str.lower()
    return df

main_df = standardize_columns(main_df)
students_df = standardize_columns(students_df)
coffee_df = standardize_columns(coffee_df)
health_df = standardize_columns(health_df)

# =========================
# 3. Select required columns
# =========================
students_df = students_df[['age', 'gender', 'academic_level', 'most_used_platform', 'affects_academic_performance']]
coffee_df = coffee_df[['age', 'gender', 'coffee_intake', 'caffeine_mg', 'sleep_quality', 'health_issues']]
health_df = health_df[['age', 'gender', 'blood_pressure', 'heart_rate', 'glucose', 'caffeine_intake']]

# =========================
# 4. Convert numeric columns
# =========================
def to_numeric(df, cols):
    for col in cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

coffee_df = to_numeric(coffee_df, ['coffee_intake', 'caffeine_mg', 'sleep_quality'])
health_df = to_numeric(health_df, ['blood_pressure', 'heart_rate', 'glucose', 'caffeine_intake'])

# =========================
# 5. Reduce duplication
# =========================
students_df = students_df.groupby(['age', 'gender'], as_index=False).agg({
    'academic_level': 'first',
    'most_used_platform': 'first',
    'affects_academic_performance': 'first'
})

coffee_df = coffee_df.groupby(['age', 'gender'], as_index=False).agg({
    'coffee_intake': 'mean',
    'caffeine_mg': 'mean',
    'sleep_quality': 'mean',
    'health_issues': 'first'
})

health_df = health_df.groupby(['age', 'gender'], as_index=False).agg({
    'blood_pressure': 'mean',
    'heart_rate': 'mean',
    'glucose': 'mean',
    'caffeine_intake': 'mean'
})

# Normalize similar columns
health_df.rename(columns={'caffeine_intake': 'caffeine_mg'}, inplace=True)

# =========================
# 6. Merge
# =========================
merged_df = main_df.merge(students_df, on=['age', 'gender'], how='left')
merged_df = merged_df.merge(coffee_df, on=['age', 'gender'], how='left')
merged_df = merged_df.merge(health_df, on=['age', 'gender'], how='left', suffixes=('', '_dup'))

merged_df = merged_df.loc[:, ~merged_df.columns.str.endswith('_dup')]

# =========================
# 7. Handle missing values (FIX WARNING HERE ✅)
# =========================
num_cols = merged_df.select_dtypes(include=['number']).columns
for col in num_cols:
    merged_df[col] = merged_df[col].fillna(merged_df[col].median())

# FIX: include both object + string
cat_cols = merged_df.select_dtypes(include=['object', 'string']).columns
for col in cat_cols:
    merged_df[col] = merged_df[col].fillna(merged_df[col].mode()[0])

# =========================
# 8. Save final dataset
# =========================
merged_df.to_csv("final.csv", index=False)

print("Final dataset saved as final.csv")

C:\Users\Victus\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\Victus\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Final dataset saved as final.csv


In [2]:
import pandas as pd

df = pd.read_csv("final.csv")

print(df.shape)
print(df.isnull().sum().sum())
print(df.head())

(1000000, 49)
1000000
   age  gender  income  daily_gaming_hours  weekly_sessions  years_gaming  \
0   51  Female    8615                3.68               22            17   
1   41  Female   39453                5.70               34            16   
2   27    Male   40466                1.58                8            22   
3   55    Male   51076                6.11               39            24   
4   20    Male   86116                3.65               17             0   

   sleep_hours  caffeine_intake  exercise_hours  stress_level  ...  \
0         5.26             1.00            0.18             3  ...   
1         9.20             0.70            1.44             8  ...   
2         7.39             2.24            3.15             3  ...   
3         7.99             1.65            2.80             1  ...   
4         7.12             1.02            1.01             2  ...   

   academic_level  most_used_platform  affects_academic_performance  \
0        Graduate      